In [2]:

import logging
import os

from dotenv import load_dotenv
load_dotenv(override=True)
logger = logging.getLogger(__name__)
def _gather_aws_context() -> str:
        try:
            import boto3
            from botocore.exceptions import BotoCoreError, ClientError

            session = boto3.session.Session()
            region = session.region_name or os.getenv("AWS_DEFAULT_REGION") or os.getenv("AWS_REGION") or ""

            lines = ["AWS ACCOUNT GROUNDING (live, fetched at pipeline start — treat as ground truth):"]

            try:
                sts = session.client("sts", region_name=region or None)
                identity = sts.get_caller_identity()
                lines.append(f"  Account ID : {identity.get('Account')}")
                lines.append(f"  Caller ARN : {identity.get('Arn')}")
            except (BotoCoreError, ClientError) as exc:
                logger.warning("aws_context.sts_failed: %s", exc)
                lines.append("  Account ID : (unavailable — could not call sts:GetCallerIdentity)")

            lines.append(f"  Region     : {region or '(not set — will rely on provider config)'}")

            try:
                ec2 = session.client("ec2", region_name=region or None)
                vpcs = ec2.describe_vpcs(Filters=[{"Name": "is-default", "Values": ["true"]}])
                default_vpc = (vpcs.get("Vpcs") or [{}])[0].get("VpcId")
                if default_vpc:
                    lines.append(f"  Default VPC: {default_vpc}")
                    subnets = ec2.describe_subnets(
                        Filters=[{"Name": "vpc-id", "Values": [default_vpc]}]
                    )
                    subnet_ids = [s["SubnetId"] for s in subnets.get("Subnets", [])]
                    if subnet_ids:
                        lines.append(f"  Default VPC subnets: {', '.join(subnet_ids)}")
                else:
                    lines.append("  Default VPC: (none found in this account/region)")

                # Fetch available Availability Zones
                azs = ec2.describe_availability_zones()
                az_names = [az["ZoneName"] for az in azs.get("AvailabilityZones", []) if az["State"] == "available"]
                if az_names:
                    lines.append(f"  Available AZs: {', '.join(az_names)}")

                # Fetch existing Key Pairs
                key_pairs = ec2.describe_key_pairs()
                kp_names = [kp["KeyName"] for kp in key_pairs.get("KeyPairs", [])]
                if kp_names:
                    lines.append(f"  Existing Key Pairs: {', '.join(kp_names)}")
                else:
                    lines.append("  Existing Key Pairs: (none found, you must generate one if needed)")
                # Fetch existing Security Groups in Default VPC
                if default_vpc:
                    sgs = ec2.describe_security_groups(Filters=[{"Name": "vpc-id", "Values": [default_vpc]}])
                    sg_info = [f"{sg['GroupName']} ({sg['GroupId']})" for sg in sgs.get("SecurityGroups", [])]
                    if sg_info:
                        lines.append(f"  Existing Security Groups: {', '.join(sg_info)}")
                    else:
                        lines.append("  Existing Security Groups: (none found)")

            except (BotoCoreError, ClientError) as exc:
                logger.warning("aws_context.ec2_failed: %s", exc)

            try:
                route53 = session.client("route53", region_name=region or None)
                zones = route53.list_hosted_zones()
                zone_info = [f"{z['Name']} (ID: {z['Id']})" for z in zones.get("HostedZones", []) if not z["Config"]["PrivateZone"]]
                if zone_info:
                    lines.append(f"  Public Route53 Zones: {', '.join(zone_info)}")
                else:
                    lines.append("  Public Route53 Zones: (none found)")
            except (BotoCoreError, ClientError) as exc:
                logger.warning("aws_context.route53_failed: %s", exc)

            lines.append(
                "\nIf an ID is provided in the context above (VPC, Subnets, Security Groups, Key Pairs, Route53 Zones), hardcode it directly in your Terraform code to avoid data source filter errors. "
                "ONLY use Terraform data sources (e.g. data \"aws_vpc\") if the required resource is marked as '(none found)' or is missing from the context.\n"
                "\nTERRAFORM GOLDEN RULES:\n"
                "1. S3 Buckets: Names must be globally unique. Always use random_id or random_pet to append a suffix to bucket names.\n"
                "2. IAM Roles/Policies: Always use name_prefix instead of name to avoid conflicts with existing roles.\n"
                "3. EC2/RDS Security Groups: Prefer using existing security groups if they match your needs, or use name_prefix when creating new ones.\n"
                "4. Circular Dependencies: Never make a Security Group depend on an EC2 instance's IP if the EC2 instance also depends on that Security Group.\n"
                "5. Hardcoding: Hardcode environment IDs (like VPCs) *only* if they are provided in the context above. NEVER hardcode full ARNs or Regions (use data.aws_caller_identity.current and data.aws_region.current instead).\n"
                "6. Stateful Resources: For databases (RDS, DynamoDB) and storage (S3), always set lifecycle { prevent_destroy = true } unless instructed otherwise.\n"
                "7. Provider Version: Use the required_providers block to specify hashicorp/aws version ~> 5.0 to ensure modern syntax is supported."
            )
            return "\n".join(lines)
        except ImportError:
            logger.info("aws_context.boto3_unavailable — skipping live AWS grounding")
            return ""
        except Exception as exc:
            logger.warning("aws_context.failed: %s", exc)
            return ""
        
result = _gather_aws_context()
print(result)

AWS ACCOUNT GROUNDING (live, fetched at pipeline start — treat as ground truth):
  Account ID : 827295473120
  Caller ARN : arn:aws:iam::827295473120:user/observation_agent
  Region     : us-east-1
  Default VPC: vpc-092ce2cf644fddc27
  Default VPC subnets: subnet-0991ae1a54e1313bc, subnet-0824abd82167b04da, subnet-0b7bc0c382430c442, subnet-0ffa581cb13e5e816, subnet-09e14f075dfbf6445, subnet-0e36ffd51a5d57df7
  Available AZs: us-east-1a, us-east-1b, us-east-1c, us-east-1d, us-east-1e, us-east-1f
  Existing Key Pairs: Chandra 05-04, Email-agent, mahesh-key, Chandra-public-key, Chandra v1
  Existing Security Groups: launch-wizard-1 (sg-0f1a7667b884947ff), launch-wizard-2 (sg-0aa4f7618c1bac288), rel-001-postgres-sg (sg-04c5590f9cb4a1946), default (sg-084543c18876a4b99), rds-postgres-access (sg-0bbca4a13e1742259)
  Public Route53 Zones: (none found)

If an ID is provided in the context above (VPC, Subnets, Security Groups, Key Pairs, Route53 Zones), hardcode it directly in your Terraform c